In [4]:
import os
import re
import sys
import json
import time
import requests
from pathlib import Path
from tqdm import tqdm
 
import pandas as pd
from openai import OpenAI

In [5]:
ROOT = Path("themes_ollama.ipynb").resolve().parent.parent
 
THEMES_OUTPUT_PATH  = ROOT / "data" / "processed" / "song_themes.json"
ARTIST_THEMES_PATH  = ROOT / "data" / "processed" / "artist_themes.json"
PROGRESS_CACHE_PATH = ROOT / "data" / "cache"    / "theme_progress.json"

In [6]:
THEMES = [
    # Romantic spectrum — the core of your corpus
    "romantic love",
    "lust / desire",
    "heartbreak / breakup",
    "longing / unrequited love",
    "toxic relationship",

    # Emotional states
    "happiness / joy",
    "anger / revenge",
    "grief / loss",
    "loneliness",
    "mental health / anxiety",

    # Identity & growth
    "self-confidence / empowerment",
    "self-discovery / coming of age",
    "girlhood / female experience",
    "body image",
    "LGBTQ",

    # Life & lifestyle
    "party / hedonism",
    "fame / celebrity life",
    "holiday / seasonal",
]
print(f"There are {len(THEMES)} themes.")

There are 18 themes.


In [7]:
def check_ollama_running() -> bool:
    """Return True if the Ollama server is reachable."""
    try:
        r = requests.get("http://localhost:11434", timeout=3)
        return r.status_code == 200
    except requests.exceptions.ConnectionError:
        return False
    
def get_ollama_client(model: str = "llama3.2") -> tuple[OpenAI, str]:
    """
    Return a configured OpenAI client pointed at the local Ollama server,
    and the model name to use.
 
    Raises a clear error if Ollama isn't running.
    """
    if not check_ollama_running():
        raise RuntimeError(
            "Ollama server is not running.\n"
            "Start it with:  ollama serve\n"
            "Then re-run this script."
        )
 
    client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama",       # required by the client lib but ignored by Ollama
    )
    return client, model

In [8]:
SYSTEM_PROMPT = """You are a music analyst specializing in lyrical themes.
Your job is to assign themes to songs from a fixed taxonomy.
You must ONLY use themes from the provided list — do not invent new ones.
Always respond with a valid JSON array of strings and nothing else.
Do not include any explanation, preamble, or markdown formatting."""
 
 
def build_classification_prompt(
    lyrics: str,
    title: str,
    artist: str,
    max_themes: int = 3,
) -> str:
    """Build the user prompt for a single song classification."""
    # Use first 200 words — enough context without hitting token limits
    lyrics_snippet = " ".join(lyrics.split()[:200])
    themes_formatted = "\n".join(f"  - {t}" for t in THEMES)
 
    return f"""Assign 1 to {max_themes} themes to this song from the list below.
 
AVAILABLE THEMES:
{themes_formatted}
 
SONG: "{title}" by {artist}
 
LYRICS:
{lyrics_snippet}
 
RULES:
- Choose only from the available themes above
- Return between 1 and {max_themes} themes
- Return a JSON array of strings only
- Example valid response: ["heartbreak / breakup", "moving on / healing"]
 
Your response:"""
 
 
def parse_llm_response(raw: str) -> list[str]:
    """
    Parse the LLM's response into a list of valid theme strings.
 
    Handles common failure modes:
      - Markdown code fences (```json ... ```)
      - Extra explanation text before/after the JSON
      - Single-quoted instead of double-quoted JSON
      - Themes not in our taxonomy (filtered out)
    """
    # Strip markdown fences
    raw = re.sub(r"```(?:json)?", "", raw).strip()
    raw = raw.strip("`").strip()
 
    # Try to find a JSON array anywhere in the response
    match = re.search(r"\[.*?\]", raw, re.DOTALL)
    if not match:
        return []
 
    try:
        parsed = json.loads(match.group())
        if not isinstance(parsed, list):
            return []
        # Only keep themes that are in our taxonomy (case-insensitive match)
        valid = [
            t for t in parsed
            if isinstance(t, str)
            and any(t.lower() == theme.lower() for theme in THEMES)
        ]
        return valid
    except json.JSONDecodeError:
        return []
 
 
def classify_song(
    client: OpenAI,
    model: str,
    lyrics: str,
    title: str,
    artist: str,
    max_themes: int = 3,
    retries: int = 2,
) -> list[str]:
    """
    Classify a single song using the Ollama model.
 
    Args:
        client:     Configured OpenAI client pointed at Ollama.
        model:      Model name (e.g. "llama3.2").
        lyrics:     Raw or lightly cleaned lyrics string.
        title:      Song title (used in the prompt for context).
        artist:     Artist name (used in the prompt for context).
        max_themes: Maximum number of themes to assign.
        retries:    Number of retry attempts on failure.
 
    Returns:
        List of theme strings from the THEMES taxonomy.
        Returns [] if classification fails after all retries.
    """
    prompt = build_classification_prompt(lyrics, title, artist, max_themes)
 
    for attempt in range(retries + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=80,
                temperature=0.0,    # deterministic — critical for consistency
            )
            raw = response.choices[0].message.content.strip()
            themes = parse_llm_response(raw)
 
            if themes:
                return themes
 
            # If parsing succeeded but returned no valid themes, retry
            if attempt < retries:
                print(f"  [retry {attempt+1}] No valid themes parsed for '{title}'. Raw: {raw[:80]}")
 
        except Exception as e:
            if attempt < retries:
                print(f"  [retry {attempt+1}] API error for '{title}': {e}")
                time.sleep(2)
            else:
                print(f"  [fail] '{title}' by {artist}: {e}")
 
    return []

In [9]:
def load_progress() -> dict:
    """
    Load previously saved classification progress.
    Keys are "artist|||title", values are lists of theme strings.
    This lets you resume interrupted runs without re-classifying songs.
    """
    if not PROGRESS_CACHE_PATH.exists():
        return {}
    with open(PROGRESS_CACHE_PATH, encoding="utf-8") as f:
        return json.load(f)
 
 
def save_progress(progress: dict) -> None:
    PROGRESS_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(PROGRESS_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)
 
 
def make_progress_key(artist: str, title: str) -> str:
    return f"{artist}|||{title}"

In [10]:
def classify_corpus(
    df: pd.DataFrame,
    lyrics_column: str = "lyrics_for_llm",
    model: str = "llama3.2",
    max_themes: int = 3,
    sleep_between: float = 0.3,
    save_every: int = 50,
    resume: bool = True,
) -> pd.DataFrame:
    """
    Classify themes for every song in the corpus using Ollama.
 
    Features:
      - Resumable: saves progress every `save_every` songs so an interrupted
        run can be continued without re-classifying from scratch.
      - Skips songs already classified in a previous run (if resume=True).
      - Saves final results to data/processed/song_themes.json.
 
    Args:
        df:             DataFrame with lyrics and metadata.
        lyrics_column:  Column containing text to classify.
                        Use "lyrics_for_llm" (light cleaning, natural language).
        model:          Ollama model name. "llama3.2" is recommended.
        max_themes:     Maximum themes per song (1-3 recommended).
        sleep_between:  Seconds to sleep between API calls.
                        0.3s is fine for local Ollama (no rate limits).
        save_every:     Save progress cache every N songs.
        resume:         If True, skip songs already in the progress cache.
 
    Returns:
        DataFrame with added "themes" column (list of strings per song).
        Also saves to THEMES_OUTPUT_PATH.
    """
    client, model = get_ollama_client(model)
    print(f"Connected to Ollama. Using model: {model}")
    print(f"Classifying {len(df)} songs into {len(THEMES)} themes...\n")
 
    # Load existing progress if resuming
    progress = load_progress() if resume else {}
    skipped  = 0
 
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Classifying"):
        key = make_progress_key(row["artist"], row["title"])
 
        # Skip already classified songs
        if resume and key in progress:
            skipped += 1
            continue
 
        lyrics = row.get(lyrics_column, "")
        if not isinstance(lyrics, str) or len(lyrics.strip()) < 20:
            progress[key] = []
            continue
 
        themes = classify_song(
            client=client,
            model=model,
            lyrics=lyrics,
            title=row["title"],
            artist=row["artist"],
            max_themes=max_themes,
        )
        progress[key] = themes
 
        # Save progress periodically
        if (i + 1) % save_every == 0:
            save_progress(progress)
 
        time.sleep(sleep_between)
 
    # Final save
    save_progress(progress)
 
    if skipped > 0:
        print(f"\nSkipped {skipped} already-classified songs (resume=True).")
 
    # Build output DataFrame
    df = df.copy()
    df["themes"] = df.apply(
        lambda r: progress.get(make_progress_key(r["artist"], r["title"]), []),
        axis=1,
    )
 
    # Save final results
    THEMES_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(THEMES_OUTPUT_PATH, orient="records", indent=2)
    print(f"\nSaved theme classifications → {THEMES_OUTPUT_PATH}")
 
    # Print coverage stats
    classified   = df["themes"].apply(lambda x: len(x) > 0).sum()
    unclassified = len(df) - classified
    print(f"Classified:   {classified}/{len(df)} songs")
    print(f"Unclassified: {unclassified} songs (empty lyrics or parse failures)")
 
    return df

In [11]:
def get_artist_theme_distribution(
    df: pd.DataFrame,
    artist: str,
    normalize: bool = True,
) -> pd.DataFrame:
    """
    Return the theme distribution for a single artist.
 
    Args:
        df:        DataFrame with a "themes" column (list of strings per row).
        artist:    Artist name to filter by.
        normalize: If True, return proportions. If False, return raw counts.
 
    Returns:
        DataFrame with columns: theme, count (or proportion).
        Sorted descending.
    """
    artist_df = df[df["artist"] == artist].copy()
    if artist_df.empty:
        raise ValueError(f"Artist '{artist}' not found in DataFrame.")
 
    # Explode themes list into one row per theme
    exploded = artist_df.explode("themes").dropna(subset=["themes"])
    exploded = exploded[exploded["themes"] != ""]
 
    counts = (
        exploded["themes"]
        .value_counts()
        .reset_index()
    )
    counts.columns = ["theme", "count"]
 
    if normalize:
        total = counts["count"].sum()
        counts["proportion"] = (counts["count"] / total).round(4)
 
    return counts
 
 
def get_all_artist_theme_distributions(
    df: pd.DataFrame,
    normalize: bool = True,
    min_songs: int = 1,
) -> pd.DataFrame:
    """
    Compute theme distributions for all artists and return as a
    wide-format DataFrame suitable for a heatmap.
 
    Rows = artists, Columns = themes, Values = proportion (or count).
 
    Args:
        df:         DataFrame with "themes" column.
        normalize:  If True, use proportions (recommended for heatmaps).
        min_songs:  Only include artists with at least this many classified songs.
 
    Returns:
        Wide-format DataFrame of shape (n_artists, n_themes).
    """
    artists = sorted(df["artist"].unique())
    rows    = {}
 
    for artist in artists:
        artist_df = df[df["artist"] == artist]
        n_classified = artist_df["themes"].apply(lambda x: len(x) > 0).sum()
 
        if n_classified < min_songs:
            continue
 
        dist = get_artist_theme_distribution(df, artist, normalize=normalize)
        rows[artist] = dict(zip(dist["theme"], dist["proportion" if normalize else "count"]))
 
    wide = pd.DataFrame(rows).T.fillna(0).round(4)
 
    # Ensure all themes are present as columns even if no artist used them
    for theme in THEMES:
        if theme not in wide.columns:
            wide[theme] = 0.0
 
    # Order columns by taxonomy order
    wide = wide[[t for t in THEMES if t in wide.columns]]
 
    return wide
 
 
def get_corpus_theme_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a summary of theme frequencies across the full corpus.
 
    Returns:
        DataFrame with columns: theme, song_count, proportion.
        Sorted descending by song_count.
    """
    exploded = df.explode("themes").dropna(subset=["themes"])
    exploded = exploded[exploded["themes"] != ""]
 
    counts = (
        exploded["themes"]
        .value_counts()
        .reset_index()
    )
    counts.columns = ["theme", "song_count"]
    counts["proportion"] = (counts["song_count"] / len(df)).round(4)
 
    return counts
 
 
def get_songs_by_theme(
    df: pd.DataFrame,
    theme: str,
    artist: str | None = None,
) -> pd.DataFrame:
    """
    Return all songs assigned to a given theme.
 
    Args:
        df:     DataFrame with "themes" column.
        theme:  Theme string — must be in THEMES taxonomy.
        artist: Optional artist filter.
 
    Returns:
        DataFrame with columns: artist, title, album, themes.
        Sorted by artist then title.
    """
    if theme not in THEMES:
        raise ValueError(
            f"'{theme}' is not in the theme taxonomy. "
            f"Valid themes: {THEMES}"
        )
 
    mask = df["themes"].apply(lambda t: theme in t if isinstance(t, list) else False)
    result = df[mask].copy()
 
    if artist:
        result = result[result["artist"] == artist]
 
    return (
        result[["artist", "title", "album", "themes"]]
        .sort_values(["artist", "title"])
        .reset_index(drop=True)
    )
 
 
def get_theme_overlap_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute a theme co-occurrence matrix — how often do pairs of themes
    appear together in the same song?
 
    Returns:
        Square DataFrame of shape (n_themes, n_themes).
        Values = number of songs where both themes appear together.
        Useful for understanding which themes cluster (e.g. lust + toxic relationship).
    """
    # Binary indicator matrix: songs × themes
    indicator = pd.DataFrame(
        {
            theme: df["themes"].apply(
                lambda t: 1 if (isinstance(t, list) and theme in t) else 0
            )
            for theme in THEMES
        }
    )
 
    # Co-occurrence = dot product of binary matrix with its transpose
    cooccurrence = indicator.T.dot(indicator)
 
    return cooccurrence

In [23]:
data_path = ROOT / "data" / "processed" / "final_songs.json"
df = pd.read_json(data_path)
df = df[df["artist"]=="Ethel Cain"]

In [20]:
model = "llama3.2"

In [24]:
themes = classify_corpus(
        df=df,
        lyrics_column="preprocessed_lyrics",
        model=model,
        resume=False,
        max_themes=1
    )

Connected to Ollama. Using model: llama3.2
Classifying 89 songs into 18 themes...



Classifying: 100%|██████████| 89/89 [01:14<00:00,  1.19it/s]


Saved theme classifications → /Users/josepomarino/Desktop/Data Projects/song_lyrics/data/processed/song_themes.json
Classified:   89/89 songs
Unclassified: 0 songs (empty lyrics or parse failures)


In [25]:
themes[["title", "themes"]]

,title,themes
1073,Ptolemaea,[longing / unrequited love]
1074,Nettles,"[longing / unrequited love, heartbreak / breakup]"
1075,American Teenager,[longing / unrequited love]
1076,Strangers,[longing / unrequited love]
1077,Sun Bleached Flies,[heartbreak / breakup]
...,...,...
1157,The Altar (Reprise),[heartbreak / breakup]
1158,Vessels,[longing / unrequited love]
1159,Apologetics (Synapse Track I),"[longing / unrequited love, heartbreak / breakup]"
1160,Colossus - Foreword,"[longing / unrequited love, mental health / an..."


In [23]:
themes[["title", "themes"]].iloc[0]

title                                                Sue me
themes    [longing / unrequited love, self-confidence / ...
Name: 291, dtype: object

In [26]:
get_artist_theme_distribution(themes, 'Ethel Cain')

,theme,count,proportion
0,longing / unrequited love,64,0.5664
1,heartbreak / breakup,31,0.2743
2,lust / desire,10,0.0885
3,toxic relationship,4,0.0354
4,grief / loss,2,0.0177
5,loneliness,1,0.0088
6,mental health / anxiety,1,0.0088
